In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, Dense, LSTM

In [2]:
df = pd.read_csv("PJME_preprocessed.csv")

In [3]:
df[
    [
        "Datetime",
        "PJME_MW",
        "Hour",
        "DayOfWeek",
        "Month",
        "IsWeekend"
    ]
].head(10)

,Datetime,PJME_MW,Hour,DayOfWeek,Month,IsWeekend
0,2002-01-08 01:00:00,29445.0,1,1,1,0
1,2002-01-08 02:00:00,28670.0,2,1,1,0
2,2002-01-08 03:00:00,28375.0,3,1,1,0
3,2002-01-08 04:00:00,28542.0,4,1,1,0
4,2002-01-08 05:00:00,29261.0,5,1,1,0
5,2002-01-08 06:00:00,31348.0,6,1,1,0
6,2002-01-08 07:00:00,35335.0,7,1,1,0
7,2002-01-08 08:00:00,37841.0,8,1,1,0
8,2002-01-08 09:00:00,37417.0,9,1,1,0
9,2002-01-08 10:00:00,36824.0,10,1,1,0


In [6]:
features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend"
]

In [7]:
df = df.dropna().reset_index(drop=True)

In [8]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [9]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [10]:
feature_columns = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend"
]

In [11]:
X_train_raw = train_df[
    feature_columns
].values

X_validation_raw = validation_df[
    feature_columns
].values

X_test_raw = test_df[
    feature_columns
].values

In [12]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [13]:
feature_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler.transform(
    X_test_raw
)

In [14]:
target_scaler = MinMaxScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler.transform(
    y_validation_raw
)

y_test_scaled = target_scaler.transform(
    y_test_raw
)

In [15]:
LOOKBACK = 48
HORIZON = 24

In [16]:
def create_sequences(
    X,
    y,
    lookback,
    horizon
):
    
    X_sequences = []
    y_sequences = []

    for i in range(
        lookback,
        len(X) - horizon + 1
    ):
        
        X_sequences.append(
            X[i - lookback:i]
        )
        
        y_sequences.append(
            y[i:i + horizon, 0]
        )

    return (
        np.array(X_sequences),
        np.array(y_sequences)
    )

In [17]:
X_train, y_train = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [18]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

In [19]:
validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [20]:
X_validation, y_validation = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [21]:
bilstm_time = Sequential([
    Bidirectional(
    LSTM(
        64
        ),
        input_shape=(LOOKBACK, 7)
    ),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [22]:
bilstm_time.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [23]:
history_time = bilstm_time.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation,
        y_validation
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - loss: 0.0077 - mae: 0.0657 - val_loss: 0.0132 - val_mae: 0.0971
Epoch 2/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0082 - mae: 0.0691 - val_loss: 0.0092 - val_mae: 0.0792
Epoch 3/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0072 - mae: 0.0649 - val_loss: 0.0061 - val_mae: 0.0641
Epoch 4/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0037 - mae: 0.0457 - val_loss: 0.0039 - val_mae: 0.0502
Epoch 5/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0040 - val_mae: 0.0498
Epoch 6/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - loss: 0.0022 - mae: 0.0351 - val_loss: 0.0040 - val_mae: 0.0492
Epoch 7/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0037 - val_mae: 0.0472
Epoch 8/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0019 - mae: 0.0323 - val_loss: 0.0034 - val_mae: 0.0444
Epoch 9/15
1792/1792 ━━━

In [24]:
y_pred_scaled = bilstm_time.predict(
    X_validation,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [25]:
y_validation_actual = target_scaler.inverse_transform(
    y_validation.reshape(-1, 1)
).reshape(
    y_validation.shape
)

In [26]:
y_pred_actual = target_scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).reshape(
    y_pred_scaled.shape
)

In [27]:
mae = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
) * 100
r2 = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [28]:
results_time = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + Time Features"
    ],
    "MAE": [mae],
    "RMSE": [rmse],
    "MAPE": [mape],
    "R2": [r2],
    "Bias": [bias]
})
results_time

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668


In [29]:
baseline = pd.DataFrame({
    "Experiment": [
        "Phase 5 bilstm-48 Baseline"
    ],
    "MAE": [2120.811443],
    "RMSE": [2889.450767],
    "MAPE": [6.455979],
    "R2": [0.801667],
    "Bias": [-1297.351713]
})

In [30]:
comparison = pd.concat(
    [
        baseline,
        results_time
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668


In [31]:
df["Lag_1"] = df["PJME_MW"].shift(1)

df["Lag_24"] = df["PJME_MW"].shift(24)

df["Lag_48"] = df["PJME_MW"].shift(48)

df["Lag_168"] = df["PJME_MW"].shift(168)

In [32]:
df[
    [
        "Datetime",
        "PJME_MW",
        "Lag_1",
        "Lag_24",
        "Lag_48",
        "Lag_168"
    ]
].head(180)

,Datetime,PJME_MW,Lag_1,Lag_24,Lag_48,Lag_168
0,2002-01-08 01:00:00,29445.0,NaN,NaN,NaN,NaN
1,2002-01-08 02:00:00,28670.0,29445.0,NaN,NaN,NaN
2,2002-01-08 03:00:00,28375.0,28670.0,NaN,NaN,NaN
3,2002-01-08 04:00:00,28542.0,28375.0,NaN,NaN,NaN
4,2002-01-08 05:00:00,29261.0,28542.0,NaN,NaN,NaN
...,...,...,...,...,...,...
175,2002-01-15 08:00:00,34375.0,32057.0,35194.0,25970.0,37841.0
176,2002-01-15 09:00:00,34143.0,34375.0,34939.0,27369.0,37417.0
177,2002-01-15 10:00:00,33509.0,34143.0,34515.0,28455.0,36824.0
178,2002-01-15 11:00:00,33178.0,33509.0,34062.0,28982.0,36504.0


In [33]:
df = df.dropna().reset_index(drop=True)

In [34]:
feature_columns_lag = [
    "PJME_MW",
    
    "Hour_sin",
    "Hour_cos",
    
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    
    "Month",
    "IsWeekend",
    
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168"
]

In [35]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [36]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [37]:
X_train_raw = train_df[
    feature_columns_lag
].values

X_validation_raw = validation_df[
    feature_columns_lag
].values

X_test_raw = test_df[
    feature_columns_lag
].values

In [38]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [39]:
feature_scaler_lag = MinMaxScaler()

X_train_scaled = feature_scaler_lag.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_lag.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_lag.transform(
    X_test_raw
)

In [40]:
target_scaler_lag = MinMaxScaler()

y_train_scaled = target_scaler_lag.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_lag.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_lag.transform(
    y_test_raw
)

In [41]:
LOOKBACK = 48
HORIZON = 24

In [42]:
X_train_lag, y_train_lag = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [43]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [44]:
X_validation_lag, y_validation_lag = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [45]:
bilstm_lag = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, 11)
    ),
    Dense(
        64,
        activation="relu"
    ),  
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [46]:
bilstm_lag.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [47]:
history_lag = bilstm_lag.fit(
    X_train_lag,
    y_train_lag,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation_lag,
        y_validation_lag
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - loss: 0.0073 - mae: 0.0631 - val_loss: 0.0133 - val_mae: 0.0977
Epoch 2/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0067 - mae: 0.0619 - val_loss: 0.0102 - val_mae: 0.0844
Epoch 3/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - loss: 0.0062 - mae: 0.0603 - val_loss: 0.0060 - val_mae: 0.0647
Epoch 4/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0042 - mae: 0.0493 - val_loss: 0.0047 - val_mae: 0.0570
Epoch 5/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0035 - mae: 0.0449 - val_loss: 0.0041 - val_mae: 0.0519
Epoch 6/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0030 - mae: 0.0411 - val_loss: 0.0038 - val_mae: 0.0495
Epoch 7/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0025 - mae: 0.0372 - val_loss: 0.0039 - val_mae: 0.0494
Epoch 8/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0023 - mae: 0.0354 - val_loss: 0.0034 - val_mae: 0.0455
Epoch 9/15
1790/1790 ━━━

In [48]:
y_pred_scaled = bilstm_lag.predict(
    X_validation_lag,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [49]:
y_validation_actual = target_scaler_lag.inverse_transform(
    y_validation_lag.reshape(-1, 1)
).reshape(
    y_validation_lag.shape
)

y_pred_actual = target_scaler_lag.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).reshape(
    y_pred_scaled.shape
)

In [50]:
mae_lag = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_lag = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_lag = np.sqrt(mse_lag)

mape_lag = mean_absolute_percentage_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
) * 100

r2_lag = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_lag = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [51]:
lag_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + Time + Lag Features"
    ],
    
    "MAE": [mae_lag],
    
    "RMSE": [rmse_lag],
    
    "MAPE": [mape_lag],
    
    "R2": [r2_lag],
    
    "Bias": [bias_lag]
})

lag_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + Time + Lag Features,1699.733155,2239.10914,5.103314,0.841634,-170.774868


In [52]:
comparison = pd.concat(
    [
        comparison,
        lag_result
    ],
    ignore_index=True
)

In [53]:
df["RollingMean_24"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["RollingMean_168"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["RollingStd_24"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["RollingStd_168"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=168)
    .std()
)

In [54]:
df[
    [
        "Datetime",
        "PJME_MW",
        "RollingMean_24",
        "RollingMean_168",
        "RollingStd_24",
        "RollingStd_168"
    ]
].tail(20)

,Datetime,PJME_MW,RollingMean_24,RollingMean_168,RollingStd_24,RollingStd_168
145036,2018-08-02 05:00:00,29854.0,39803.833333,35786.809524,6965.040633,6786.448887
145037,2018-08-02 06:00:00,31197.0,39861.041667,35800.077381,6873.026278,6772.561274
145038,2018-08-02 07:00:00,33182.0,39909.458333,35813.125000,6804.549431,6761.494335
145039,2018-08-02 08:00:00,35645.0,39943.500000,35825.994048,6767.105765,6754.370726
145040,2018-08-02 09:00:00,37810.0,39981.166667,35839.607143,6739.347844,6751.670950
145041,2018-08-02 10:00:00,39902.0,40024.708333,35855.708333,6721.016198,6753.133834
145042,2018-08-02 11:00:00,42189.0,40080.125000,35874.970238,6713.996874,6760.070306
145043,2018-08-02 12:00:00,43954.0,40158.166667,35895.976190,6727.729404,6774.244281
145044,2018-08-02 13:00:00,45372.0,40222.791667,35916.255952,6757.613188,6793.324049
145045,2018-08-02 14:00:00,46534.0,40284.666667,35934.392857,6799.292698,6814.570888


In [55]:
df = df.dropna().reset_index(drop=True)

In [56]:
feature_columns_rolling = [
    "PJME_MW",
    
    "Hour_sin",
    "Hour_cos",
    
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    
    "Month",
    "IsWeekend",
    
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [57]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [58]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [59]:
X_train_raw = train_df[
    feature_columns_rolling
].values

X_validation_raw = validation_df[
    feature_columns_rolling
].values

X_test_raw = test_df[
    feature_columns_rolling
].values

In [60]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [61]:
feature_scaler_rolling = MinMaxScaler()

X_train_scaled = feature_scaler_rolling.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_rolling.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_rolling.transform(
    X_test_raw
)

In [62]:
target_scaler_rolling = MinMaxScaler()

y_train_scaled = target_scaler_rolling.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_rolling.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_rolling.transform(
    y_test_raw
)

In [63]:
X_train_rolling, y_train_rolling = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [64]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [65]:
X_validation_rolling, y_validation_rolling = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [66]:
bilstm_rolling = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, 15)
    ),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [67]:
bilstm_rolling.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [68]:
history_rolling = bilstm_rolling.fit(
    X_train_rolling,
    y_train_rolling,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation_rolling,
        y_validation_rolling
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - loss: 0.0065 - mae: 0.0606 - val_loss: 0.0151 - val_mae: 0.1048
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0056 - mae: 0.0570 - val_loss: 0.0121 - val_mae: 0.0939
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0049 - mae: 0.0535 - val_loss: 0.0061 - val_mae: 0.0663
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0031 - mae: 0.0415 - val_loss: 0.0040 - val_mae: 0.0520
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0023 - mae: 0.0354 - val_loss: 0.0030 - val_mae: 0.0438
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0020 - mae: 0.0330 - val_loss: 0.0028 - val_mae: 0.0417
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0019 - mae: 0.0320 - val_loss: 0.0027 - val_mae: 0.0407
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0018 - mae: 0.0310 - val_loss: 0.0026 - val_mae: 0.0399
Epoch 9/15
1788/1788 ━━━

In [69]:
y_pred_scaled = bilstm_rolling.predict(
    X_validation_rolling,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [70]:
y_validation_actual = (
    target_scaler_rolling
    .inverse_transform(
        y_validation_rolling.reshape(-1, 1)
    )
    .reshape(y_validation_rolling.shape)
)

y_pred_actual = (
    target_scaler_rolling
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [71]:
mae_rolling = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_rolling = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_rolling = np.sqrt(mse_rolling)

mape_rolling = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_rolling = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_rolling = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [72]:
rolling_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + Time + Lag + Rolling Features"
    ],
    
    "MAE": [mae_rolling],
    
    "RMSE": [rmse_rolling],
    
    "MAPE": [mape_rolling],
    
    "R2": [r2_rolling],
    
    "Bias": [bias_rolling]
})

rolling_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.14593,0.854453,206.55184


In [73]:
comparison = pd.concat(
    [
        comparison,
        rolling_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840


In [74]:
comparison.sort_values(
    by="R2",
    ascending=False
)
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    bilstm-48 + Time + Lag + Rolling Features
MAE                                         1678.903587
RMSE                                        2189.625023
MAPE                                            5.14593
R2                                             0.854453
Bias                                          206.55184
Name: 3, dtype: object


In [75]:
best_features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [76]:
train_df = df.iloc[:validation_start].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[test_start:].copy()

In [77]:
X_train_raw = train_df[best_features].values

X_validation_raw = validation_df[best_features].values

X_test_raw = test_df[best_features].values

In [78]:
y_train_raw = train_df[["PJME_MW"]].values

y_validation_raw = validation_df[["PJME_MW"]].values

y_test_raw = test_df[["PJME_MW"]].values

In [79]:
feature_scaler_dropout = MinMaxScaler()

X_train_scaled = feature_scaler_dropout.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_dropout.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_dropout.transform(
    X_test_raw
)

In [80]:
target_scaler_dropout = MinMaxScaler()

y_train_scaled = target_scaler_dropout.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_dropout.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_dropout.transform(
    y_test_raw
)

In [81]:
X_train_dropout, y_train_dropout = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [82]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [83]:
X_validation_dropout, y_validation_dropout = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [84]:
n_features = X_train_dropout.shape[2]

print("Number of features:", n_features)

Number of features: 15


In [85]:
from tensorflow.keras.layers import Bidirectional, Dense, Dropout

In [86]:
bilstm_dropout = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [87]:
bilstm_dropout.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [88]:
history_dropout = bilstm_dropout.fit(
    X_train_dropout,
    y_train_dropout,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_dropout,
        y_validation_dropout
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - loss: 0.0116 - mae: 0.0814 - val_loss: 0.0100 - val_mae: 0.0846
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0070 - mae: 0.0643 - val_loss: 0.0057 - val_mae: 0.0632
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0044 - mae: 0.0508 - val_loss: 0.0044 - val_mae: 0.0549
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0037 - mae: 0.0465 - val_loss: 0.0034 - val_mae: 0.0470
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0033 - mae: 0.0439 - val_loss: 0.0029 - val_mae: 0.0425
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0031 - mae: 0.0424 - val_loss: 0.0028 - val_mae: 0.0414
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0030 - mae: 0.0416 - val_loss: 0.0028 - val_mae: 0.0408
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0027 - val_mae: 0.0400
Epoch 9/15
1788/1788 ━━━

In [89]:
y_pred_scaled = bilstm_dropout.predict(
    X_validation_dropout,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [90]:
y_validation_actual = (
    target_scaler_dropout
    .inverse_transform(
        y_validation_dropout.reshape(-1, 1)
    )
    .reshape(y_validation_dropout.shape)
)

y_pred_actual = (
    target_scaler_dropout
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [91]:
mae_dropout = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_dropout = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_dropout = np.sqrt(mse_dropout)

mape_dropout = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_dropout = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_dropout = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [92]:
dropout_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + Dropout"
    ],
    "MAE": [mae_dropout],
    "RMSE": [rmse_dropout],
    "MAPE": [mape_dropout],
    "R2": [r2_dropout],
    "Bias": [bias_dropout]
})

In [93]:
comparison = pd.concat(
    [
        comparison,
        dropout_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005


In [94]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713


In [95]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    bilstm-48 + Time + Lag + Rolling Features
MAE                                         1678.903587
RMSE                                        2189.625023
MAPE                                            5.14593
R2                                             0.854453
Bias                                          206.55184
Name: 3, dtype: object


In [96]:
best_features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [97]:
train_df = df.iloc[:validation_start].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[test_start:].copy()

In [98]:
X_train_raw = train_df[best_features].values

X_validation_raw = validation_df[best_features].values

X_test_raw = test_df[best_features].values

In [99]:
y_train_raw = train_df[["PJME_MW"]].values

y_validation_raw = validation_df[["PJME_MW"]].values

y_test_raw = test_df[["PJME_MW"]].values

In [100]:
feature_scaler_bn = MinMaxScaler()

X_train_scaled = feature_scaler_bn.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_bn.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_bn.transform(
    X_test_raw
)

In [101]:
target_scaler_bn = MinMaxScaler()

y_train_scaled = target_scaler_bn.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_bn.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_bn.transform(
    y_test_raw
)

In [102]:
X_train_bn, y_train_bn = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [103]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [104]:
X_validation_bn, y_validation_bn = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [105]:
n_features = X_train_bn.shape[2]

print("Number of features:", n_features)

Number of features: 15


In [106]:
from tensorflow.keras.layers import (
    Bidirectional,
    Dense,
    Dropout,
    BatchNormalization
)

In [107]:
bilstm_bn = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    BatchNormalization(),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    BatchNormalization(),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [108]:
bilstm_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [109]:
history_bn = bilstm_bn.fit(
    X_train_bn,
    y_train_bn,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_bn,
        y_validation_bn
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 36s 19ms/step - loss: 0.0879 - mae: 0.1826 - val_loss: 0.0142 - val_mae: 0.0966
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0121 - mae: 0.0871 - val_loss: 0.0190 - val_mae: 0.1111
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0114 - mae: 0.0845 - val_loss: 0.0132 - val_mae: 0.0941
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0115 - mae: 0.0853 - val_loss: 0.0161 - val_mae: 0.1021
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0117 - mae: 0.0855 - val_loss: 0.0146 - val_mae: 0.0984
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0114 - mae: 0.0848 - val_loss: 0.0149 - val_mae: 0.0996
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0113 - mae: 0.0844 - val_loss: 0.0154 - val_mae: 0.1009
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0111 - mae: 0.0838 - val_loss: 0.0164 - val_mae: 0.1018
Epoch 9/15
1788/1788 ━━━

In [110]:
y_pred_scaled = bilstm_bn.predict(
    X_validation_bn,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [111]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_bn.reshape(-1, 1)
    )
    .reshape(y_validation_bn.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [112]:
mae_bn = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_bn = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_bn = np.sqrt(mse_bn)

mape_bn = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_bn = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_bn = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [113]:
bn_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + Batch Normalization"
    ],
    "MAE": [mae_bn],
    "RMSE": [rmse_bn],
    "MAPE": [mape_bn],
    "R2": [r2_bn],
    "Bias": [bias_bn]
})

bn_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + Batch Normalization,4996.75374,6366.270092,14.756804,-0.230361,-1562.486567


In [114]:
comparison = pd.concat(
    [
        comparison,
        bn_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567


In [115]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567


In [116]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    bilstm-48 + Time + Lag + Rolling Features
MAE                                         1678.903587
RMSE                                        2189.625023
MAPE                                            5.14593
R2                                             0.854453
Bias                                          206.55184
Name: 3, dtype: object


In [117]:
X_train_optimizer = X_train_bn
y_train_optimizer = y_train_bn

X_validation_optimizer = X_validation_bn
y_validation_optimizer = y_validation_bn

In [118]:
bilstm_rmsprop = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [119]:
from tensorflow.keras.optimizers import RMSprop

In [120]:
rmsprop_optimizer = RMSprop(
    learning_rate=0.001
)

In [121]:
bilstm_rmsprop.compile(
    optimizer=rmsprop_optimizer,
    loss="mse",
    metrics=["mae"]
)

In [122]:
history_rmsprop = bilstm_rmsprop.fit(
    X_train_optimizer,
    y_train_optimizer,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_optimizer,
        y_validation_optimizer
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 17ms/step - loss: 0.0076 - mae: 0.0644 - val_loss: 0.0103 - val_mae: 0.0853
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0039 - mae: 0.0477 - val_loss: 0.0083 - val_mae: 0.0760
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0035 - mae: 0.0450 - val_loss: 0.0077 - val_mae: 0.0729
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 19ms/step - loss: 0.0033 - mae: 0.0434 - val_loss: 0.0072 - val_mae: 0.0707
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0031 - mae: 0.0425 - val_loss: 0.0068 - val_mae: 0.0680
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0030 - mae: 0.0418 - val_loss: 0.0065 - val_mae: 0.0670
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0030 - mae: 0.0413 - val_loss: 0.0063 - val_mae: 0.0657
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0061 - val_mae: 0.0646
Epoch 9/15
1788/1788 ━━━

In [123]:
y_pred_scaled = bilstm_rmsprop.predict(
    X_validation_optimizer,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [124]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_optimizer.reshape(-1, 1)
    )
    .reshape(y_validation_optimizer.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [125]:
mae_rmsprop = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_rmsprop = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_rmsprop = np.sqrt(
    mse_rmsprop
)
mape_rmsprop = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_rmsprop = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_rmsprop = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [126]:
rmsprop_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + RMSprop"
    ],
    "MAE": [
        mae_rmsprop
    ],
    "RMSE": [
        rmse_rmsprop
    ],
    "MAPE": [
        mape_rmsprop
    ],
    "R2": [
        r2_rmsprop
    ],
    "Bias": [
        bias_rmsprop
    ]
})

rmsprop_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411


In [127]:
comparison = pd.concat(
    [
        comparison,
        rmsprop_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411


In [128]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567


In [129]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    bilstm-48 + Time + Lag + Rolling Features
MAE                                         1678.903587
RMSE                                        2189.625023
MAPE                                            5.14593
R2                                             0.854453
Bias                                          206.55184
Name: 3, dtype: object


In [130]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [131]:
bilstm_lr = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [132]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [133]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [134]:
rmsprop_optimizer = RMSprop(
    learning_rate=0.001
)

In [135]:
bilstm_lr.compile(
    optimizer=rmsprop_optimizer,
    loss="mse",
    metrics=["mae"]
)

In [136]:
history_lr = bilstm_lr.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - loss: 0.0079 - mae: 0.0655 - val_loss: 0.0094 - val_mae: 0.0814 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0040 - mae: 0.0485 - val_loss: 0.0077 - val_mae: 0.0727 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0036 - mae: 0.0457 - val_loss: 0.0071 - val_mae: 0.0702 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0034 - mae: 0.0440 - val_loss: 0.0068 - val_mae: 0.0680 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0032 - mae: 0.0430 - val_loss: 0.0065 - val_mae: 0.0668 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 19ms/step - loss: 0.0031 - mae: 0.0422 - val_loss: 0.0064 - val_mae: 0.0665 - learning_rate: 0.0010
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0030 - mae: 0.0418 - val_loss: 0.0062 - val_mae: 0.0648 - 

In [137]:
y_pred_scaled = bilstm_lr.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [138]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [139]:
mae_lr = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_lr = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_lr = np.sqrt(mse_lr)
mape_lr = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_lr = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_lr = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [140]:
lr_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + RMSprop + LR Scheduling"
    ],
    "MAE": [mae_lr],
    "RMSE": [rmse_lr],
    "MAPE": [mape_lr],
    "R2": [r2_lr],
    "Bias": [bias_lr]
})

lr_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241


In [141]:
comparison = pd.concat(
    [
        comparison,
        lr_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241


In [142]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [143]:
from tensorflow.keras.callbacks import EarlyStopping

In [144]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [145]:
bilstm_early = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [146]:
optimizer = RMSprop(
    learning_rate=0.001
)

In [147]:
bilstm_early.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

In [148]:
history_early = bilstm_early.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler,
        early_stopping
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0075 - mae: 0.0635 - val_loss: 0.0098 - val_mae: 0.0825 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0039 - mae: 0.0477 - val_loss: 0.0080 - val_mae: 0.0741 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0034 - mae: 0.0447 - val_loss: 0.0071 - val_mae: 0.0695 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0032 - mae: 0.0432 - val_loss: 0.0067 - val_mae: 0.0669 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0031 - mae: 0.0422 - val_loss: 0.0064 - val_mae: 0.0654 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - loss: 0.0030 - mae: 0.0415 - val_loss: 0.0061 - val_mae: 0.0636 - learning_rate: 0.0010
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0058 - val_mae: 0.0628 - 

In [149]:
y_pred_scaled = bilstm_early.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [150]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [151]:
mae_early = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_early = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_early = np.sqrt(mse_early)

mape_early = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_early = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_early = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [152]:
early_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + RMSprop + LR Scheduling + Early Stopping"
    ],
    "MAE": [mae_early],
    "RMSE": [rmse_early],
    "MAPE": [mape_early],
    "R2": [r2_early],
    "Bias": [bias_early]
})

early_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.46421,8.617806,0.644417,128.045998


In [153]:
comparison = pd.concat(
    [
        comparison,
        early_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241
8,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.464210,8.617806,0.644417,128.045998


In [154]:
bilstm_batch32 = Sequential([
    Bidirectional(
    LSTM(
        64
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [155]:
optimizer_batch32 = RMSprop(
    learning_rate=0.001
)

In [156]:
bilstm_batch32.compile(
    optimizer=optimizer_batch32,
    loss="mse",
    metrics=["mae"]
)

In [157]:
lr_scheduler_batch32 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [158]:
early_stopping_batch32 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [159]:
history_batch32 = bilstm_batch32.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=32,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler_batch32,
        early_stopping_batch32
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 38s 10ms/step - loss: 0.0057 - mae: 0.0560 - val_loss: 0.0133 - val_mae: 0.1001 - learning_rate: 0.0010
Epoch 2/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 37s 10ms/step - loss: 0.0035 - mae: 0.0448 - val_loss: 0.0117 - val_mae: 0.0931 - learning_rate: 0.0010
Epoch 3/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 37s 10ms/step - loss: 0.0032 - mae: 0.0427 - val_loss: 0.0109 - val_mae: 0.0895 - learning_rate: 0.0010
Epoch 4/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 38s 11ms/step - loss: 0.0030 - mae: 0.0416 - val_loss: 0.0102 - val_mae: 0.0869 - learning_rate: 0.0010
Epoch 5/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 38s 11ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0100 - val_mae: 0.0859 - learning_rate: 0.0010
Epoch 6/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 38s 11ms/step - loss: 0.0029 - mae: 0.0405 - val_loss: 0.0100 - val_mae: 0.0861 - learning_rate: 0.0010
Epoch 7/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 38s 11ms/step - loss: 0.0028 - mae: 0.0401 - val_loss: 0.0100 - val_mae: 0.0858 - 

In [160]:
y_pred_scaled = bilstm_batch32.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [161]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [162]:
mae_batch32 = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_batch32 = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_batch32 = np.sqrt(mse_batch32)

mape_batch32 = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_batch32 = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_batch32 = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [163]:
batch32_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + RMSprop + LR Scheduling + Early Stopping + Batch Size 32"
    ],
    "MAE": [mae_batch32],
    "RMSE": [rmse_batch32],
    "MAPE": [mape_batch32],
    "R2": [r2_batch32],
    "Bias": [bias_batch32]
})

batch32_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + RMSprop + LR Scheduling + Early St...,3685.190514,4319.39628,11.478469,0.433619,719.797693


In [164]:
comparison = pd.concat(
    [
        comparison,
        batch32_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241
8,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.464210,8.617806,0.644417,128.045998
9,bilstm-48 + RMSprop + LR Scheduling + Early St...,3685.190514,4319.396280,11.478469,0.433619,719.797693


In [165]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [166]:
bilstm_layers = Sequential([
    Bidirectional(
    LSTM(
        64,
        return_sequences=True
    ),
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    Bidirectional(
    LSTM(
        32
    )
    ),
    Dropout(0.2),
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [167]:
optimizer_layers = RMSprop(
    learning_rate=0.001
)

In [168]:
bilstm_layers.compile(
    optimizer=optimizer_layers,
    loss="mse",
    metrics=["mae"]
)

In [169]:
lr_scheduler_layers = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [170]:
early_stopping_layers = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [171]:
history_layers = bilstm_layers.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler_layers,
        early_stopping_layers
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 69s 37ms/step - loss: 0.0069 - mae: 0.0617 - val_loss: 0.0122 - val_mae: 0.0931 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 68s 38ms/step - loss: 0.0033 - mae: 0.0442 - val_loss: 0.0092 - val_mae: 0.0804 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 69s 38ms/step - loss: 0.0028 - mae: 0.0399 - val_loss: 0.0082 - val_mae: 0.0757 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 69s 38ms/step - loss: 0.0026 - mae: 0.0382 - val_loss: 0.0076 - val_mae: 0.0732 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 69s 39ms/step - loss: 0.0024 - mae: 0.0372 - val_loss: 0.0073 - val_mae: 0.0717 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 69s 39ms/step - loss: 0.0023 - mae: 0.0364 - val_loss: 0.0071 - val_mae: 0.0703 - learning_rate: 0.0010
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 68s 38ms/step - loss: 0.0023 - mae: 0.0358 - val_loss: 0.0067 - val_mae: 0.0689 - 

In [172]:
y_pred_scaled = bilstm_layers.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


In [173]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [174]:
mae_layers = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_layers = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_layers = np.sqrt(mse_layers)

mape_layers = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_layers = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_layers = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [175]:
layers_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 + 2 Layers (64-32)"
    ],
    "MAE": [mae_layers],
    "RMSE": [rmse_layers],
    "MAPE": [mape_layers],
    "R2": [r2_layers],
    "Bias": [bias_layers]
})

layers_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 + 2 Layers (64-32),3019.188058,3611.859266,9.449334,0.603974,569.573621


In [176]:
comparison = pd.concat(
    [
        comparison,
        layers_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241
8,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.464210,8.617806,0.644417,128.045998
9,bilstm-48 + RMSprop + LR Scheduling + Early St...,3685.190514,4319.396280,11.478469,0.433619,719.797693


In [177]:
import keras_tuner as kt

In [178]:
def build_bilstm_model(hp):

    model = Sequential()

    bilstm_units = hp.Choice(
        "bilstm_units",
        values=[32, 64, 128]
    )

    dense_units = hp.Choice(
        "dense_units",
        values=[32, 64, 128]
    )

    dropout_rate = hp.Choice(
        "dropout_rate",
        values=[0.1, 0.2, 0.3]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0005, 0.001, 0.002]
    )

    model.add(
        Bidirectional(
        LSTM(
            units=bilstm_units,
            input_shape=(LOOKBACK, n_features)
        )
    )
    )

    model.add(
        Dropout(dropout_rate)
    )

    model.add(
        Dense(
            units=dense_units,
            activation="relu"
        )
    )

    model.add(
        Dropout(dropout_rate)
    )

    model.add(
        Dense(HORIZON)
    )

    optimizer = RMSprop(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [179]:
tuner = kt.RandomSearch(
    build_bilstm_model,
    objective="val_loss",
    max_trials=5,
    executions_per_trial=1,
    directory="bilstm_tuning",
    project_name="electricity_demand_bilstm",
    overwrite=True
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [180]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
bilstm_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
dense_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
dropout_rate (Choice)
{'default': 0.1, 'conditions': [], 'values': [0.1, 0.2, 0.3], 'ordered': True}
learning_rate (Choice)
{'default': 0.0005, 'conditions': [], 'values': [0.0005, 0.001, 0.002], 'ordered': True}


In [181]:
lr_scheduler_tuning = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [182]:
early_stopping_tuning = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [183]:
tuner.search(
    X_train_lr,
    y_train_lr,

    epochs=15,

    batch_size=64,

    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),

    callbacks=[
        lr_scheduler_tuning,
        early_stopping_tuning
    ],

    shuffle=False,

    verbose=1
)

Trial 5 Complete [00h 04m 23s]
val_loss: 0.0049774013459682465

Best val_loss So Far: 0.004304866772145033
Total elapsed time: 00h 34m 43s


In [184]:
best_hp = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

In [185]:
print(
    "Best bilstm units:",
    best_hp.get("bilstm_units")
)

print(
    "Best Dense units:",
    best_hp.get("dense_units")
)

print(
    "Best Dropout:",
    best_hp.get("dropout_rate")
)

print(
    "Best Learning Rate:",
    best_hp.get("learning_rate")
)

Best bilstm units: 32
Best Dense units: 128
Best Dropout: 0.2
Best Learning Rate: 0.001


In [186]:
best_bilstm_tuned = tuner.get_best_models(
    num_models=1
)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [187]:
y_pred_scaled = best_bilstm_tuned.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [188]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [189]:
mae_tuned = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_tuned = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_tuned = np.sqrt(mse_tuned)
mape_tuned = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_tuned = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_tuned = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [190]:
tuned_result = pd.DataFrame({
    "Experiment": [
        "bilstm-48 Hyperparameter Tuning"
    ],
    "MAE": [mae_tuned],
    "RMSE": [rmse_tuned],
    "MAPE": [mape_tuned],
    "R2": [r2_tuned],
    "Bias": [bias_tuned]
})

tuned_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,bilstm-48 Hyperparameter Tuning,2530.466742,3114.248771,7.833001,0.705579,322.257024


In [191]:
comparison = pd.concat(
    [
        comparison,
        tuned_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
5,bilstm-48 + Batch Normalization,4996.753740,6366.270092,14.756804,-0.230361,-1562.486567
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241
8,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.464210,8.617806,0.644417,128.045998
9,bilstm-48 + RMSprop + LR Scheduling + Early St...,3685.190514,4319.396280,11.478469,0.433619,719.797693


In [192]:
best_index = comparison["R2"].idxmax()

best_bilstm_result = comparison.loc[
    best_index
]

print("BEST bilstm RESULT")
print(best_bilstm_result)

BEST bilstm RESULT
Experiment    bilstm-48 + Time + Lag + Rolling Features
MAE                                         1678.903587
RMSE                                        2189.625023
MAPE                                            5.14593
R2                                             0.854453
Bias                                          206.55184
Name: 3, dtype: object


In [193]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,bilstm-48 + Time + Lag + Rolling Features,1678.903587,2189.625023,5.145930,0.854453,206.551840
2,bilstm-48 + Time + Lag Features,1699.733155,2239.109140,5.103314,0.841634,-170.774868
4,bilstm-48 + Dropout,1749.704313,2288.787654,5.306878,0.840972,-251.300005
1,bilstm-48 + Time Features,1707.111232,2325.090452,5.033614,0.824272,-453.275668
0,Phase 5 bilstm-48 Baseline,2120.811443,2889.450767,6.455979,0.801667,-1297.351713
11,bilstm-48 Hyperparameter Tuning,2530.466742,3114.248771,7.833001,0.705579,322.257024
8,bilstm-48 + RMSprop + LR Scheduling + Early St...,2799.251553,3422.464210,8.617806,0.644417,128.045998
7,bilstm-48 + RMSprop + LR Scheduling,2814.531897,3426.320066,8.725403,0.643616,363.498241
6,bilstm-48 + RMSprop,2822.337342,3437.507823,8.693338,0.641285,196.379411
10,bilstm-48 + 2 Layers (64-32),3019.188058,3611.859266,9.449334,0.603974,569.573621


In [194]:
comparison.to_csv(
    "phase6_bilstm_final_comparison.csv",
    index=False
)

In [195]:
best_bilstm_tuned.save(
    "best_bilstm_phase6.keras"
)